# G3000: daily maximum and minimum temperature

This notebook loads the G3000 wide JSON export, calculates the daily maximum and minimum of `outdoor_temperature`, and plots both values. The JSON currently contains 5-minute readings; the daily table groups readings by calendar day in the timestamps' local time.

If you move or rename the data file, update `DATA_PATH` in the next cell. Dependencies are managed by `uv` from `pyproject.toml`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("data/GW3000A-WIFI1573/ecowitt_GW3000A-WIFI1573_wide.json")
required_columns = {"timestamp", "outdoor_temperature"}

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Set DATA_PATH to the G3000 wide JSON file."
    )

weather = pd.read_json(DATA_PATH, convert_dates=False)
missing_columns = required_columns.difference(weather.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="raise")
weather["outdoor_temperature"] = pd.to_numeric(
    weather["outdoor_temperature"], errors="coerce"
)

daily_temperature = (
    weather.dropna(subset=["outdoor_temperature"])
    .set_index("timestamp")["outdoor_temperature"]
    .resample("D")
    .agg(min_temperature="min", max_temperature="max")
    .dropna(how="all")
)

print(f"Loaded {len(weather):,} readings from {DATA_PATH}")
print(f"Daily groups: {len(daily_temperature)}")
daily_temperature

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    daily_temperature.index,
    daily_temperature["max_temperature"],
    marker="o",
    label="Daily maximum",
    color="firebrick",
)
ax.plot(
    daily_temperature.index,
    daily_temperature["min_temperature"],
    marker="o",
    label="Daily minimum",
    color="royalblue",
)
ax.fill_between(
    daily_temperature.index,
    daily_temperature["min_temperature"],
    daily_temperature["max_temperature"],
    color="slategray",
    alpha=0.15,
    label="Daily range",
)

ax.set_title("G3000 outdoor temperature: daily minimum and maximum")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()